In [1]:
!pip install duckdb

In [2]:
import duckdb
import pandas as pd

In [3]:
con = duckdb.connect()

In [6]:
customer_df = pd.read_csv("customer_features.csv")

In [7]:
con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM customer_df
""")

In [8]:
con.execute("""
SELECT *
FROM customers
LIMIT 5
""").df()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,...,satisfaction_flag,promo_dependency,purchase_frequency_score,purchase_amount_norm,previous_purchases_norm,frequency_score_norm,customer_value_score,value_tier,loyalty_a,loyalty_b
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Needs Improvement,High,4,0.4125,0.265306,0.75,0.421122,Medium,Not Loyal,Not Loyal
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Needs Improvement,High,4,0.5500,0.020408,0.75,0.378163,Low,Not Loyal,Not Loyal
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Needs Improvement,High,5,0.6625,0.448980,1.00,0.644592,High,Not Loyal,Not Loyal
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Needs Improvement,High,5,0.8750,0.979592,1.00,0.941837,High,Not Loyal,Not Loyal
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Needs Improvement,High,1,0.3625,0.612245,0.00,0.389898,Low,Not Loyal,Not Loyal


# SQL Business Analysis

## Project Objective

This notebook answers key business questions using SQL to identify high-value customers, evaluate promotion dependency, uncover customer behavior patterns, and recommend retention strategies.

**Database:** DuckDB

**Dataset:** customer_features.csv

# Query 1: Customer Overview

## Business Objective

Understand the overall customer base before beginning customer segmentation.

### Expected Insight

This query summarizes the customer base by calculating average age,
purchase amount, previous purchases, and review ratings.

In [9]:
con.sql("""
SELECT
    COUNT(*) AS total_customers,
    ROUND(AVG(age),2) AS average_age,
    ROUND(AVG(purchase_amount_usd),2) AS average_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS average_previous_purchases,
    ROUND(AVG(review_rating),2) AS average_review_rating
FROM customers;
""").df()

,total_customers,average_age,average_purchase_amount,average_previous_purchases,average_review_rating
0,3900,44.07,59.76,25.35,3.75


## Insight

The dataset consists of **3,900 customers** with an average age of **44 years**.

On average, each customer spends **$59.76** per transaction and has made **25.35 previous purchases**, indicating a customer base with a reasonable level of repeat purchasing.

The average review rating of **3.75/5** suggests that customers are generally satisfied, although there is room for improvement in the overall shopping experience.

## Business Recommendation

The customer base demonstrates healthy purchasing activity and moderate customer satisfaction. Rather than adopting a one-size-fits-all marketing strategy, the business should segment customers based on purchasing behavior and value to develop more targeted retention campaigns.

# Query 2: Customer Value Tier Distribution

## Business Objective

Determine how customers are distributed across the engineered value tiers and compare their purchasing behavior.

In [10]:
con.sql("""
SELECT
    value_tier,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(review_rating),2) AS avg_review_rating
FROM customers
GROUP BY value_tier
ORDER BY avg_value_score DESC;
""").df()

,value_tier,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases,avg_review_rating
0,High,1299,0.696,77.09,36.18,3.77
1,Medium,1301,0.494,59.85,25.23,3.74
2,Low,1300,0.298,42.37,14.66,3.74


## Insight

Customers are almost evenly distributed across the three engineered value tiers due to quantile-based segmentation.

However, purchasing behavior differs significantly:

- High-value customers spend an average of **$77.09** and have completed **36.18 previous purchases**.
- Medium-value customers spend **$59.85** with **25.23 previous purchases**.
- Low-value customers spend only **$42.37** and average **14.66 previous purchases**.

Interestingly, the average review rating remains relatively consistent across all three segments (approximately **3.74–3.77**), suggesting that customer satisfaction alone does not explain differences in customer value.



## Business Recommendation

Since satisfaction levels are similar across all customer segments, increasing customer value should focus on encouraging more frequent purchases and increasing transaction value rather than solely improving customer satisfaction. High-value customers should receive personalized loyalty benefits, while low-value customers can be targeted with campaigns aimed at increasing purchase frequency.

# Query 3: Loyalty vs Customer Value

## Business Objective

Evaluate whether loyal customers, as defined during feature engineering, generate greater business value than non-loyal customers.

In [13]:
con.sql("""
SELECT
    loyalty_a,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(review_rating),2) AS avg_review_rating
FROM customers
GROUP BY loyalty_a;
""").df()

,loyalty_a,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases,avg_review_rating
0,Not Loyal,3439,0.473,59.61,23.77,3.75
1,Loyal,461,0.666,60.89,37.13,3.77


## Insight

The loyalty segmentation successfully differentiates customers based on value.

Although loyal customers represent only **461 customers (11.8%)** of the customer base, they have a substantially higher average Customer Value Score (**0.666**) compared to non-loyal customers (**0.473**).

The average purchase amount is only slightly higher for loyal customers ($60.89 vs. $59.61), suggesting that loyalty is driven more by **consistent purchasing behavior and repeat engagement** than by individual transaction size.

## Business Recommendation

The business should prioritize retaining the loyal customer segment through exclusive membership benefits, early access to new collections, and personalized rewards rather than relying on blanket discounts. Since loyal customers are relatively few but generate higher long-term value, improving their retention is likely to deliver a stronger return on investment than acquiring new customers.

# Query 4: Promotion Dependency Analysis

## Business Objective

Determine whether customers who rely on promotions differ in value and purchasing behavior from customers who do not.

In [14]:
con.sql("""
SELECT
    promo_dependency,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(review_rating),2) AS avg_review_rating
FROM customers
GROUP BY promo_dependency;
""").df()

,promo_dependency,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases,avg_review_rating
0,High,1677,0.497,59.28,25.74,3.74
1,Low,2223,0.495,60.13,25.06,3.76


## Insight

Customers with **High Promotion Dependency** and **Low Promotion Dependency** exhibit remarkably similar purchasing behavior.

| Metric | High Dependency | Low Dependency |
|--------|----------------:|---------------:|
| Customer Value Score | 0.497 | 0.495 |
| Average Purchase | $59.28 | $60.13 |
| Previous Purchases | 25.74 | 25.06 |
| Review Rating | 3.74 | 3.76 |

The differences across all key metrics are minimal, indicating that promotion dependency alone does not appear to be a strong indicator of customer value or purchasing behavior within this dataset.

## Business Recommendation

The company should avoid making broad decisions, such as reducing promotions for all customers, based solely on promotion dependency. Instead, promotions should be analyzed alongside customer value, loyalty, product category, and geographic location to develop more targeted marketing strategies.

# Query 5: Promotion Dependency by Customer Value Tier

## Business Objective

Determine whether promotion dependency varies across different customer value segments.

In [15]:
con.sql("""
SELECT
    value_tier,
    promo_dependency,
    COUNT(*) AS customer_count,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases
FROM customers
GROUP BY value_tier, promo_dependency
ORDER BY value_tier, promo_dependency;
""").df()

,value_tier,promo_dependency,customer_count,avg_purchase_amount,avg_previous_purchases
0,High,High,565,76.17,36.69
1,High,Low,734,77.80,35.78
2,Low,High,557,42.71,14.80
3,Low,Low,743,42.11,14.56
4,Medium,High,555,58.72,25.59
5,Medium,Low,746,60.69,24.96


## Insight

Promotion dependency is fairly balanced across all three customer value tiers, with a slightly higher proportion of customers classified as **Low Promotion Dependency**.

Within each value tier, customers with **Low Promotion Dependency** consistently demonstrate slightly higher average purchase amounts:

| Value Tier | High Promo | Low Promo |
|------------|-----------:|----------:|
| High | $76.17 | **$77.80** |
| Medium | $58.72 | **$60.69** |
| Low | $42.71 | $42.11 |

The differences are relatively small, suggesting that customer value is driven primarily by purchasing behavior rather than promotion usage.

## Business Recommendation

The findings suggest that blanket discount strategies are unlikely to substantially increase customer value. Instead, the business should prioritize personalized promotions based on customer value tier. High-value customers should receive exclusive loyalty rewards rather than frequent discounts, while medium- and low-value customers can be targeted with promotions designed to increase purchase frequency and encourage progression into higher value segments.

# Query 6: Purchase Frequency Analysis

## Business Objective

Understand how purchasing frequency influences customer value and identify the purchasing patterns associated with high-value customers.

In [16]:
con.sql("""
SELECT
    frequency_of_purchases,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases
FROM customers
GROUP BY frequency_of_purchases
ORDER BY avg_value_score DESC;
""").df()

,frequency_of_purchases,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases
0,Weekly,539,0.597,58.97,25.77
1,Bi-Weekly,547,0.548,60.69,24.79
2,Fortnightly,542,0.543,59.05,25.27
3,Monthly,553,0.495,59.33,25.28
4,Quarterly,563,0.461,59.98,26.85
5,Every 3 Months,584,0.446,60.08,24.96
6,Annually,572,0.393,60.17,24.56


## Insight

Purchase frequency has a strong positive relationship with customer value.

Customers who shop **weekly** have the highest average Customer Value Score (0.597), while customers who shop **annually** have the lowest (0.393). This indicates that frequent engagement contributes more to long-term customer value than single high-value purchases.

Interestingly, the average purchase amount remains relatively constant (around $59–$60) across all purchase frequencies. This suggests that the difference in customer value is driven primarily by **repeat purchasing behavior** rather than higher spending per transaction.

## Business Recommendation

The business should focus on increasing purchase frequency through retention initiatives such as loyalty programs, replenishment reminders, personalized recommendations, and targeted email campaigns. Encouraging customers to purchase more often is likely to generate greater long-term value than attempting to increase the value of individual transactions.

# Query 7: Top 10 Highest Value Customers

## Business Objective

Identify the highest-value customers who contribute the greatest long-term business value and should be prioritized for retention initiatives.

In [17]:
con.sql("""
SELECT
    customer_id,
    customer_value_score,
    value_tier,
    loyalty_a,
    purchase_amount_usd,
    previous_purchases,
    frequency_of_purchases
FROM customers
ORDER BY customer_value_score DESC
LIMIT 10;
""").df()

,customer_id,customer_value_score,value_tier,loyalty_a,purchase_amount_usd,previous_purchases,frequency_of_purchases
0,993,0.995000,High,Not Loyal,99,50,Weekly
1,1848,0.991837,High,Loyal,100,49,Weekly
2,2446,0.980000,High,Loyal,96,50,Weekly
3,3582,0.975000,High,Loyal,95,50,Weekly
4,2289,0.955000,High,Loyal,91,50,Weekly
5,456,0.950000,High,Not Loyal,100,50,Fortnightly
6,1069,0.945510,High,Not Loyal,94,47,Weekly
7,4,0.941837,High,Not Loyal,90,49,Weekly
8,1738,0.940000,High,Loyal,98,50,Bi-Weekly
9,1512,0.938673,High,Not Loyal,91,48,Weekly


# Query 8: Subscription Status vs Customer Value

## Business Objective

Determine whether customers with an active subscription generate greater business value than non-subscribers.

In [18]:
con.sql("""
SELECT
    subscription_status,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(review_rating),2) AS avg_review_rating
FROM customers
GROUP BY subscription_status;
""").df()

,subscription_status,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases,avg_review_rating
0,No,2847,0.494,59.87,25.08,3.75
1,Yes,1053,0.502,59.49,26.08,3.75


# Query 9: Payment Method Analysis

## Business Objective

Identify whether customer value differs across payment methods and determine if payment preferences are associated with stronger customer engagement.

In [19]:
con.sql("""
SELECT
    payment_method,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases
FROM customers
GROUP BY payment_method
ORDER BY avg_value_score DESC;
""").df()

,payment_method,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases
0,Debit Card,636,0.502,60.92,25.56
1,Credit Card,671,0.501,60.07,25.59
2,Cash,670,0.499,59.70,25.25
3,PayPal,677,0.493,59.25,25.51
4,Venmo,634,0.492,58.95,25.65
5,Bank Transfer,612,0.489,59.71,24.50


# Query 10: Shipping Type Analysis

## Business Objective

Analyze customer behavior across different shipping methods to identify whether shipping preferences are associated with higher customer value.

In [20]:
con.sql("""
SELECT
    shipping_type,
    COUNT(*) AS customer_count,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases
FROM customers
GROUP BY shipping_type
ORDER BY avg_value_score DESC;
""").df()

,shipping_type,customer_count,avg_value_score,avg_purchase_amount,avg_previous_purchases
0,2-Day Shipping,627,0.507,60.73,26.11
1,Express,646,0.502,60.48,25.50
2,Standard,654,0.497,58.46,26.23
3,Free Shipping,675,0.493,60.41,24.74
4,Store Pickup,650,0.491,59.89,24.80
5,Next Day Air,648,0.487,58.63,24.77


# Business Question 1

## Who are the genuinely loyal customers versus discount-driven customers?

### Business Objective

Identify customer segments that are naturally loyal versus those whose purchases appear to depend on promotions. This helps determine where promotions can be reduced without affecting retention.

In [26]:
con.sql("""
SELECT
    loyalty_a,
    promo_dependency,
    COUNT(*) AS customer_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_customers,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount
FROM customers
GROUP BY loyalty_a, promo_dependency
ORDER BY avg_value_score DESC;
""").df()

,loyalty_a,promo_dependency,customer_count,percentage_of_customers,avg_value_score,avg_previous_purchases,avg_purchase_amount
0,Loyal,Low,461,11.82,0.666,37.13,60.89
1,Not Loyal,High,1677,43.00,0.497,25.74,59.28
2,Not Loyal,Low,1762,45.18,0.450,21.90,59.93


The analysis identified three distinct customer segments based on loyalty and promotion dependency.

The **Loyal + Low Promotion Dependency** segment represents only **11.82%** of the customer base (461 customers) but has the highest average Customer Value Score (**0.666**) and the highest average number of previous purchases (**37.13**). This indicates that these customers generate strong long-term value without relying on promotional incentives.

In contrast, **88.18%** of customers are classified as not loyal. Among them, 43.00% are highly promotion-dependent, while 45.18% are not promotion-dependent but still demonstrate lower customer value than the loyal segment.

The company should prioritize retaining the loyal customer segment through personalized loyalty programs, exclusive rewards, and early access to new products instead of frequent discounts.

For the large non-loyal segment, different strategies should be adopted. Promotion-dependent customers can be targeted with carefully designed promotional campaigns, while non-loyal customers with low promotion dependency should be engaged through personalized recommendations and initiatives that encourage repeat purchases.

This segmented approach can improve customer retention while optimizing promotional spending.

# Business Question 2

## Product Category Performance

### Business Objective

Determine which product categories are associated with the highest-value and most loyal customers.

In [22]:
con.sql("""
SELECT
    category,
    COUNT(*) AS customers,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount
FROM customers
GROUP BY category
ORDER BY avg_value_score DESC;
""").df()

,category,customers,avg_value_score,avg_previous_purchases,avg_purchase_amount
0,Footwear,599,0.499,25.23,60.26
1,Accessories,1240,0.499,25.73,59.84
2,Clothing,1737,0.496,25.20,60.03
3,Outerwear,324,0.481,24.96,57.17


The analysis shows that customer value is relatively consistent across most product categories.

Footwear and Accessories have the highest average Customer Value Score (0.499), closely followed by Clothing (0.496). Outerwear records the lowest average Customer Value Score (0.481), along with the lowest average purchase amount ($57.17).

Despite these differences, the variation across categories is relatively small. This suggests that product category alone is not a major driver of customer value, and purchasing behavior has a stronger influence on long-term customer value.

The company should avoid allocating marketing budgets solely based on product category performance, as customer value remains relatively similar across most categories.

Instead, customer segmentation strategies should prioritize behavioral factors such as purchase frequency, loyalty, and previous purchasing history. However, the Outerwear category should be investigated further to identify potential opportunities to improve customer engagement and increase average spending.

# Business Question 3

## Geographic Analysis

### Business Objective

Identify locations that generate the highest-value customers and greatest purchasing activity.

In [23]:
con.sql("""
SELECT
    location,
    COUNT(*) AS customers,
    ROUND(AVG(customer_value_score),3) AS avg_value_score,
    ROUND(AVG(purchase_amount_usd),2) AS avg_purchase_amount,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases
FROM customers
GROUP BY location
ORDER BY avg_value_score DESC
LIMIT 15;
""").df()

,location,customers,avg_value_score,avg_purchase_amount,avg_previous_purchases
0,Alaska,72,0.565,67.60,28.10
1,Arizona,65,0.554,66.55,28.37
2,Pennsylvania,74,0.552,66.57,27.42
3,Hawaii,65,0.532,57.72,29.17
4,Illinois,92,0.525,61.05,26.60
5,Wyoming,71,0.524,60.69,28.24
6,Iowa,69,0.523,60.88,27.61
7,Tennessee,77,0.520,61.97,25.96
8,Nevada,87,0.519,63.38,26.03
9,Utah,71,0.516,62.58,27.17


## Insight

The geographic analysis reveals noticeable differences in customer value across locations.

Alaska has the highest average Customer Value Score (0.565), followed by Arizona (0.554) and Pennsylvania (0.552). These locations also exhibit relatively high average purchase amounts and previous purchase counts, indicating stronger customer engagement and repeat purchasing behavior.

Although the differences between locations are not extreme, certain states consistently outperform others in terms of customer value, suggesting that geographic factors may influence purchasing behavior.

## Business Recommendation

The company should prioritize marketing and customer retention initiatives in high-performing locations such as Alaska, Arizona, and Pennsylvania to maximize return on investment.

At the same time, lower-performing regions should be analyzed separately to identify potential barriers to customer engagement, such as product preferences, promotional effectiveness, or regional buying patterns. Regional marketing strategies may be more effective than a uniform nationwide approach.

In [24]:
con.sql("""
SELECT
    gender,
    subscription_status,
    promo_dependency,
    frequency_of_purchases,
    COUNT(*) AS customers,
    ROUND(AVG(customer_value_score),3) AS avg_value_score
FROM customers
WHERE value_tier='High'
GROUP BY
    gender,
    subscription_status,
    promo_dependency,
    frequency_of_purchases
ORDER BY avg_value_score DESC
LIMIT 15;
""").df()

,gender,subscription_status,promo_dependency,frequency_of_purchases,customers,avg_value_score
0,Male,Yes,High,Weekly,87,0.723
1,Male,No,Low,Weekly,84,0.718
2,Male,No,High,Weekly,43,0.717
3,Male,No,Low,Fortnightly,51,0.716
4,Male,No,Low,Bi-Weekly,58,0.715
5,Male,No,High,Fortnightly,43,0.714
6,Female,No,Low,Weekly,87,0.711
7,Female,No,Low,Fortnightly,68,0.706
8,Female,No,Low,Bi-Weekly,74,0.702
9,Male,Yes,High,Bi-Weekly,64,0.702


In [25]:
con.sql("""
SELECT
    value_tier,
    loyalty_a,
    promo_dependency,
    COUNT(*) AS customers,
    ROUND(AVG(customer_value_score),3) AS avg_value_score
FROM customers
GROUP BY
    value_tier,
    loyalty_a,
    promo_dependency
ORDER BY
    value_tier DESC,
    avg_value_score DESC;
""").df()

,value_tier,loyalty_a,promo_dependency,customers,avg_value_score
0,Medium,Loyal,Low,123,0.513
1,Medium,Not Loyal,High,555,0.493
2,Medium,Not Loyal,Low,623,0.492
3,Low,Loyal,Low,10,0.387
4,Low,Not Loyal,High,557,0.299
5,Low,Not Loyal,Low,733,0.296
6,High,Loyal,Low,328,0.732
7,High,Not Loyal,High,565,0.697
8,High,Not Loyal,Low,406,0.666


## Insight

Among customers classified as **High Value**, the highest Customer Value Scores are most commonly associated with **weekly purchasing frequency**, regardless of gender or subscription status.

Several of the highest-ranked customer profiles purchase weekly and have Customer Value Scores above **0.70**, indicating that purchase frequency is a stronger differentiator than demographic characteristics such as gender.

The results also show that both subscribed and non-subscribed customers appear in the top-performing segments, suggesting that subscription status alone is not a reliable predictor of high customer value.

## Business Recommendation

The company should prioritize strategies that encourage more frequent purchases, such as personalized product recommendations, replenishment reminders, loyalty rewards, and targeted engagement campaigns.

Rather than designing campaigns primarily around gender or subscription status, marketing efforts should focus on behavioral patterns, particularly increasing purchase frequency, as this is more strongly associated with high customer value.

# Key Findings

## 1. Customer Value
High-value customers are distinguished primarily by repeat purchasing behavior rather than higher transaction values.

## 2. Loyalty
Only a small proportion of customers are classified as loyal, but they contribute the highest long-term value and purchase most frequently.

## 3. Promotions
Promotion dependency alone does not significantly influence customer value, suggesting that blanket discount strategies may not be effective.

## 4. Geography
Certain locations, such as Alaska and Arizona, exhibit higher average customer value and may benefit from targeted retention efforts.

## 5. Customer Strategy
Behavioral segmentation based on purchase frequency and loyalty is likely to be more effective than demographic segmentation for improving customer retention.